**PCD 1st Assignment**

In [1]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    colab_path = '/content/drive/MyDrive/Colab Notebooks'

    if os.path.exists(colab_path):
        os.chdir(colab_path)
        print("Direktori aktif:", os.getcwd())
    else:
        print("Folder tidak ditemukan")
else:
    print("Direktori aktif lokal:", os.getcwd())

Mounted at /content/drive
Direktori aktif: /content/drive/MyDrive/Colab Notebooks


Algorithm

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Fungsi bantu
def load_image(path):
    """Membaca citra dan mengonversi ke array NumPy RGB."""
    img = Image.open(path).convert('RGB')
    return np.array(img)

def show_images(titles, images, figsize=(15, 5), main_title=None):
    """Menampilkan beberapa citra sejajar dalam satu baris."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if main_title:
        fig.suptitle(main_title, fontsize=13, fontweight='bold')
    for ax, title, img in zip(axes, titles, images):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Algoritma down sampling
def down_sample(img, scale, method='average'):
    h, w, c = img.shape
    new_h, new_w = h // scale, w // scale
    out = np.zeros((new_h, new_w, c), dtype=np.uint8)

    for i in range(new_h):
        for j in range(new_w):
            block = img[i*scale:(i+1)*scale, j*scale:(j+1)*scale, :]

            if method == 'max':
                out[i, j, :] = np.max(block, axis=(0, 1))
            elif method == 'average':
                out[i, j, :] = np.mean(block, axis=(0, 1))
            elif method == 'median':
                out[i, j, :] = np.median(block, axis=(0, 1))

    return out

# Algoritma up sampling
def up_sample_nn(img, scale):
    h, w, c = img.shape
    new_h, new_w = h * scale, w * scale
    out = np.zeros((new_h, new_w, c), dtype=np.uint8)

    for i in range(new_h):
        for j in range(new_w):
            out[i, j, :] = img[i // scale, j // scale, :]

    return out

def up_sample_bilinear(img, scale):
    h, w, c = img.shape
    new_h, new_w = int(h * scale), int(w * scale)
    out = np.zeros((new_h, new_w, c), dtype=np.uint8)

    for i in range(new_h):
        for j in range(new_w):
            x, y = i / scale, j / scale

            x1, y1 = int(x), int(y)
            x2, y2 = min(x1 + 1, h - 1), min(y1 + 1, w - 1)

            dx, dy = x - x1, y - y1

            for k in range(c):
                val = (img[x1, y1, k] * (1 - dx) * (1 - dy) +
                       img[x2, y1, k] * dx * (1 - dy) +
                       img[x1, y2, k] * (1 - dx) * dy +
                       img[x2, y2, k] * dx * dy)
                out[i, j, k] = int(val)

    return out

def up_sample_bicubic(img, scale):
    def cubic_weight(x):
        x = abs(x)
        if x <= 1:
            return 1.5 * x**3 - 2.5 * x**2 + 1
        elif x < 2:
            return -0.5 * x**3 + 2.5 * x**2 - 4 * x + 2
        return 0

    h, w, c = img.shape
    new_h, new_w = int(h * scale), int(w * scale)
    out = np.zeros((new_h, new_w, c), dtype=np.uint8)

    for i in range(new_h):
        for j in range(new_w):
            x, y = i / scale, j / scale
            x_int, y_int = int(x), int(y)
            dx, dy = x - x_int, y - y_int

            for channel in range(c):
                pixel_val = 0
                for m in range(-1, 3):
                    for n in range(-1, 3):
                        xi = min(max(x_int + m, 0), h - 1)
                        yj = min(max(y_int + n, 0), w - 1)

                        weight = cubic_weight(m - dx) * cubic_weight(n - dy)
                        pixel_val += img[xi, yj, channel] * weight

                out[i, j, channel] = np.clip(pixel_val, 0, 255)

    return out

Citra Uji

Digunakan 3 citra dengan karakteristik berbeda untuk melihat pengaruh down sampling dan up sampling:
- **pemandangan.jpg** — citra alami dengan gradasi warna halus, minim detail tajam
- **banyakdetail.jpg** — citra dengan tekstur/detail tinggi
- **banyaktext.jpg** — citra yang mengandung teks, sensitif terhadap perubahan resolusi

In [3]:
# Path citra uji dengan karakteristik berbeda
image_paths = {
    'Pemandangan': 'pemandangan.jpg',
    'Banyak Detail': 'banyakdetail.jpg',
    'Banyak Teks': 'banyaktext.jpg'
}

images = {name: load_image(path) for name, path in image_paths.items()}

for name, img in images.items():
    print(f"{name}: resolusi asli {img.shape}")

Pemandangan: resolusi asli (404, 720, 3)
Banyak Detail: resolusi asli (463, 662, 3)
Banyak Teks: resolusi asli (414, 738, 3)


Down Sampling

In [4]:
# Down sampling untuk setiap citra uji
down_results = {}

for name, img in images.items():
    down_max = down_sample(img, 2, method='max')
    down_avg = down_sample(img, 2, method='average')
    down_med = down_sample(img, 2, method='median')

    down_results[name] = {'max': down_max, 'average': down_avg, 'median': down_med}

    print(f"{name}: resolusi setelah down sampling {down_avg.shape}")

    show_images(
        ['Original', 'Down: Max', 'Down: Average', 'Down: Median'],
        [img, down_max, down_avg, down_med],
        figsize=(16, 5),
        main_title=name
    )

Output hidden; open in https://colab.research.google.com to view.

Up Sampling dari Hasil Down Sampling

In [5]:
# Rekonstruksi citra dari hasil down sampling (average)
for name, img in images.items():
    down_avg = down_results[name]['average']

    up_nn = up_sample_nn(down_avg, 2)
    up_bil = up_sample_bilinear(down_avg, 2)
    up_bic = up_sample_bicubic(down_avg, 2)

    print(f"{name}: resolusi direkonstruksi menjadi {up_nn.shape}")

    show_images(
        ['Input (Down Average)', 'Up: Nearest Neighbor', 'Up: Bilinear', 'Up: Bicubic'],
        [down_avg, up_nn, up_bil, up_bic],
        figsize=(16, 5),
        main_title=name
    )

Output hidden; open in https://colab.research.google.com to view.

Up Sampling dari Citra Asli

In [6]:
# Upsampling dengan gambar original
for name, img in images.items():
    up_nn = up_sample_nn(img, 2)
    up_bil = up_sample_bilinear(img, 2)
    up_bic = up_sample_bicubic(img, 2)

    print(f"{name}: resolusi setelah up sampling dari citra asli {up_nn.shape}")

    show_images(
        ['Original', 'Up: Nearest Neighbor', 'Up: Bilinear', 'Up: Bicubic'],
        [img, up_nn, up_bil, up_bic],
        figsize=(16, 5),
        main_title=name
    )

Output hidden; open in https://colab.research.google.com to view.